In [ ]:
# Importar librerias
import duckdb
import pandas as pd
import statsforecast


In [ ]:
# Iniciar la conexión (sin abrir el "almacén" p1.duckdb)
con = duckdb.connect()

In [ ]:
# Cargar el parquet a un df()
df = con.sql("""
        SELECT * FROM '../data/processed/ventas_mensuales.parquet'
    """).df()

In [10]:
# Revisar carga
print(df.shape)
print(df.dtypes)

(3000, 3)
ds           datetime64[us]
unique_id             int64
y                   float64
dtype: object


In [14]:
# Analizar df, para decidir como dividir train / test
df.nunique()

ds             60
unique_id      50
y            2829
dtype: int64

In [ ]:
print("Fecha de inicio:", df.ds.min())
print("Fecha de fin:", df.ds.max())

Fecha de inicio: 2013-01-01 00:00:00
Fecha de fin: 2017-12-01 00:00:00


In [33]:
# Revisar la diferencia de dias entre meses
pd.Series(df.ds.unique()).diff()

0        NaT
1    31 days
2    28 days
3    31 days
4    30 days
5    31 days
6    30 days
7    31 days
8    31 days
9    30 days
10   31 days
11   30 days
12   31 days
13   31 days
14   28 days
15   31 days
16   30 days
17   31 days
18   30 days
19   31 days
20   31 days
21   30 days
22   31 days
23   30 days
24   31 days
25   31 days
26   28 days
27   31 days
28   30 days
29   31 days
30   30 days
31   31 days
32   31 days
33   30 days
34   31 days
35   30 days
36   31 days
37   31 days
38   29 days
39   31 days
40   30 days
41   31 days
42   30 days
43   31 days
44   31 days
45   30 days
46   31 days
47   30 days
48   31 days
49   31 days
50   28 days
51   31 days
52   30 days
53   31 days
54   30 days
55   31 days
56   31 days
57   30 days
58   31 days
59   30 days
dtype: timedelta64[us]

In [ ]:
# Calendario con el rango completo, para revisar que este completo el set
calendario_2013_2017 = pd.date_range(df.ds.min(), df.ds.max(), freq="MS")

In [39]:
# Set vacio, entonces fechas de "df" completas
set(calendario_2013_2017) - set(df.ds)

set()

In [ ]:
# Corte 1
train_1 = df[df.ds <= "2016-12-01"]
test_2017_Q1 = df[(df.ds >= "2017-01-01") & (df.ds <= "2017-03-01")]

In [47]:
# Comprobación del corte 1
print(train_1.ds.max())
print(test_2017_Q1.ds.min())

2016-12-01 00:00:00
2017-01-01 00:00:00


In [50]:
# Modelo naive forecast
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

c:\dev\scm-lab\.venv\Lib\site-packages\statsforecast\models.py:3811: SyntaxWarning: invalid escape sequence '\h'
  $$\hat{y}_t = \\frac{\hat{z}_t}{\hat{p}_t}$$
c:\dev\scm-lab\.venv\Lib\site-packages\statsforecast\models.py:3989: SyntaxWarning: invalid escape sequence '\h'
  $$\hat{y}_t = \\frac{\hat{z}_t}{\hat{p}_t}$$
c:\dev\scm-lab\.venv\Lib\site-packages\statsforecast\models.py:4137: SyntaxWarning: invalid escape sequence '\h'
  $$\hat{y}_t = \\frac{\hat{z}_t}{\hat{p}_t}$$
c:\dev\scm-lab\.venv\Lib\site-packages\statsforecast\models.py:4477: SyntaxWarning: invalid escape sequence '\e'
  \end{cases}


In [ ]:
sf = StatsForecast(models=[SeasonalNaive(season_length=12)], freq="MS")
pron_1 = sf.forecast(df=train_1, h=3)
pron_1

,unique_id,ds,SeasonalNaive
0,1,2017-01-01,5008.0
1,1,2017-02-01,5160.0
2,1,2017-03-01,6624.0
3,2,2017-01-01,13580.0
4,2,2017-02-01,13881.0
...,...,...,...
145,49,2017-02-01,6906.0
146,49,2017-03-01,8718.0
147,50,2017-01-01,15122.0
148,50,2017-02-01,15526.0


In [ ]:
# Comprobación y entendimiento del Naive Seasonal forecast,
df[(df.unique_id == 1) & (df.ds == "2016-01-01")]

,ds,unique_id,y
1800,2016-01-01,1,5008.0


In [ ]:
resultados = pron_1.merge(test_2017_Q1, on=["ds", "unique_id"])
resultados

,unique_id,ds,SeasonalNaive,y
0,1,2017-01-01,5008.0,5236.0
1,1,2017-02-01,5160.0,5299.0
2,1,2017-03-01,6624.0,6744.0
3,2,2017-01-01,13580.0,13481.0
4,2,2017-02-01,13881.0,14056.0
...,...,...,...,...
145,49,2017-02-01,6906.0,6973.0
146,49,2017-03-01,8718.0,9374.0
147,50,2017-01-01,15122.0,15480.0
148,50,2017-02-01,15526.0,15520.0


In [67]:
resultados["Error"] = resultados["SeasonalNaive"] - resultados["y"]

In [ ]:
sesgo = resultados["Error"].sum() / resultados["y"].sum()
print(round(sesgo * 100, 1))

wape = resultados["Error"].abs().sum() / resultados["y"].sum()
print(round(wape * 100, 1))

-2.6
2.7
